# 2025-010D: a Falcon 9 second stage → amateur astrometry → lunar impact

On 2026-08-05 the Falcon 9 second stage from the January 2025 Blue Ghost /
HAKUTO-R launch struck the Moon near crater Einstein — the first lunar
impactor whose provenance, mass, and dimensions were known in advance.
This notebook fits the final nine-night arc of the *public* astrometry
(the MPC does not serve artificial objects — see the README for the
data's provenance and the observers who made it), measures the
stage's area-to-mass ratio from solar radiation pressure, and
propagates into the Moon.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Empyrean-Dynamics/empyrean-scenarios/blob/main/2025-010D/main.ipynb)


In [1]:
%pip install -q empyrean==0.10.0rc0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/moeyensj/projects/empyrean/empyrean-rc/empyrean-py/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. The committed public astrometry

402 observations compiled by Project Pluto (Bill Gray), public domain,
transcribed field-for-field into the ADES PSV the reader consumes.
The final arc: nine nights, five stations, four continents.


In [2]:
from pathlib import Path

import empyrean
import numpy as np
from empyrean import (
    CartesianOrbits,
    Epochs,
    EventConfig,
    ODConfig,
    Origin,
    SolveFor,
    SRPParams,
    TimeScale,
)
from empyrean.od.result import (
    OriginPolicy,
    OriginPolicyMode,
    WeightingConfig,
    WeightingPreset,
)

empyrean.initialize()
HERE = Path("2025-010D") if Path("2025-010D").exists() else Path(".")
obs, _ = empyrean.read_ades(str(HERE / "astrometry.psv"))
final_arc = obs.apply_mask([t >= "2026-07-01" for t in obs.obs_time.to_pylist()])
len(obs), len(final_arc)

(402, 74)

## 2. Geocentric fit + AMR refine

`EXPLICIT`/`EARTH` origin policy (heliocentric Gauss is unphysical for
an Earth-orbiting stage), residual-matched 0.3″ weighting (the file
carries no per-observation sigmas), then the refine that measures the
area-to-mass ratio — for a hollow 4-tonne cylinder, SRP is the
dominant model term, ~30× any natural asteroid's.


In [3]:
geocentric = ODConfig(
    origin=OriginPolicy(mode=OriginPolicyMode.EXPLICIT, origin=Origin.EARTH),
    weighting=WeightingConfig(preset=WeightingPreset.NONE, default_sigma_arcsec=0.3),
)
fit = empyrean.determine(final_arc, config=geocentric).single()
s = fit.summary
print(
    f'chi2/dof {s.reduced_chi2:.3f}  RMS {s.rms_ra_arcsec:.3f}"/{s.rms_dec_arcsec:.3f}"  ({s.num_selected}/{s.num_obs})'
)

chi2/dof 0.540  RMS 0.323"/0.279"  (74/74)


In [4]:
srp = SRPParams.from_kwargs(amrat=[0.008], cr=[1.0], amrat_variance=[1.0e-4])
primed = CartesianOrbits.from_kwargs(
    orbit_id=fit.orbit.orbit_id.to_pylist(),
    object_id=fit.orbit.object_id.to_pylist(),
    coordinates=fit.orbit.coordinates,
    srp=srp,
)
refined = empyrean.refine(
    primed,
    final_arc,
    config=ODConfig(
        solve_for_flags=SolveFor(amrat="solved"),
        origin=OriginPolicy(mode=OriginPolicyMode.EXPLICIT, origin=Origin.EARTH),
        weighting=WeightingConfig(
            preset=WeightingPreset.NONE, default_sigma_arcsec=0.3
        ),
    ),
)
sc = refined.solved_covariance
amrat = refined.orbit.srp.amrat.to_numpy(zero_copy_only=False)[0]
sigma = float(np.sqrt(sc.matrix[sc.amrat_slot, sc.amrat_slot]))
print(f"Fitted AMR = {amrat:.4f} ± {sigma:.4f} m²/kg")
print("Gray (same 74-obs arc): 0.0079 ± 0.0017 m²/kg")

Fitted AMR = 0.0029 ± 0.0011 m²/kg
Gray (same 74-obs arc): 0.0079 ± 0.0017 m²/kg


## 3. Into the Moon

Propagate through 2026-08-05 and read the impact off the event
stream — epoch and selenographic coordinates, against Gray's
benchmark fit, JPL's radar-informed solution, and the confirmed event.


In [5]:
epochs = Epochs.from_kwargs(
    mjd=[65610.0 + 0.05 * i for i in range(120)],
    scale=TimeScale.TDB.value,
)
prop = empyrean.propagate(
    refined.orbit, epochs, events=EventConfig(body_filter=[Origin.MOON])
)
imp = prop.events.impacts
for i in range(len(imp)):
    utc = (
        Epochs.from_mjd(
            [imp.epoch.to_numpy(zero_copy_only=False)[i]], scale=TimeScale.TDB.value
        )
        .to_utc()
        .to_iso()[0]
    )
    lat = imp.latitude_deg.to_numpy(zero_copy_only=False)[i]
    lon = imp.longitude_deg.to_numpy(zero_copy_only=False)[i] % 360.0
    print(f"Empyrean:  {utc}  at {lat:.3f}°N {lon:.3f}°E")
print("Gray:      2026-08-05T06:35:42.45Z at 19.577°N 266.630°E (74-obs arc)")
print("JPL:       2026-08-05T06:35:40Z ± 9 s at 19.507°N 266.7°E (#GA1A2/21)")
print("Confirmed: 2026-08-05 ~06:35 UTC near crater Einstein")

Empyrean:  2026-08-05T06:35:40.169156925Z  at 19.514°N 266.646°E
Gray:      2026-08-05T06:35:42.45Z at 19.577°N 266.630°E (74-obs arc)
JPL:       2026-08-05T06:35:40Z ± 9 s at 19.507°N 266.7°E (#GA1A2/21)
Confirmed: 2026-08-05 ~06:35 UTC near crater Einstein


## 4. Why there is no grand unified fit

Score the final-arc orbit against the *whole* public record: the
residuals climb five orders of magnitude through two lunar encounters
and a tumbling body whose effective area changed. Every arc earns its
own fit — which is exactly how Project Pluto published them.


In [6]:
ev = empyrean.evaluate(refined.orbit, obs)
r = ev.observations
t_mjd = r.epoch_mjd_tdb.to_numpy(zero_copy_only=False)
res_ra = r.ra_residual.to_numpy(zero_copy_only=False)
res_dec = r.dec_residual.to_numpy(zero_copy_only=False)
for label, lo, hi in [
    ("own arc   (2026 Jul-Aug)", 61244.0, 61300.0),
    ("Apr-May 2026", 61135.0, 61190.0),
    ("Dec 2025", 61020.0, 61040.0),
    ("Jan 2025 discovery", 60680.0, 60700.0),
]:
    m = (t_mjd >= lo) & (t_mjd < hi)
    if m.any():
        rms = float(np.sqrt(np.nanmean(res_ra[m] ** 2 + res_dec[m] ** 2)))
        print(f'{label:26s} RMS {rms:>12.2f}"')

own arc   (2026 Jul-Aug)   RMS         0.41"
Apr-May 2026               RMS       165.29"
Dec 2025                   RMS      1679.28"
Jan 2025 discovery         RMS    470490.62"
